In [1]:
%pip install pandas numpy matplotlib --quiet

Note: you may need to restart the kernel to use updated packages.


In [69]:
# Import necessary libraries
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'svg'

print("Libraries imported successfully!")

# Connect to the SQLite database
conn = sqlite3.connect("inventory.db")

# Create a cursor object to interact with the database
cursor = conn.cursor()

# Function to run a SQL query and return the results as a DataFrame
def run_query_to_df(cursor, query):
    cursor.execute(query)
    rows = cursor.fetchall()
    
    # Get column names from cursor
    columns = []
    for col in cursor.description:
        columns.append(col[0])

    # Convert to DataFrame
    df = pd.DataFrame(rows, columns = columns)

    return df

Libraries imported successfully!


In [68]:
query = """
SELECT oi.quantity, s.days_scheduled, s.days_actual,
       (s.days_actual / s.days_scheduled) * 100 AS late_risk_rate
FROM orders AS o
JOIN customers  AS c  ON o.customer_id = c.id
JOIN shipping   AS s  ON o.id = s.order_id
JOIN order_items AS oi ON o.id = oi.order_id
ORDER BY oi.quantity DESC, s.late_delivery_risk
LIMIT 10
"""

# Call the function to run the query and display results
result = run_query_to_df(cursor, query)
result

,quantity,days_scheduled,days_actual,late_risk_rate
0,23,4.0,4.0,100.0
1,22,4.0,3.0,75.0
2,22,4.0,6.0,150.0
3,22,1.0,2.0,200.0
4,22,4.0,5.0,125.0
5,21,4.0,2.0,50.0
6,21,4.0,4.0,100.0
7,21,4.0,2.0,50.0
8,21,0.0,0.0,NaN
9,21,1.0,2.0,200.0


In [65]:
query = """
SELECT round(AVG(days_actual/days_scheduled)*100,2) AS avg_late_risk_rate FROM orders AS o

JOIN customers as c
ON o.customer_id = c.id

JOIN shipping as s
ON o.id = s.order_id
"""

# Call the function to run the query and display results
result = run_query_to_df(cursor, query)
result

,avg_late_risk_rate
0,136.69


In [92]:
query = """
SELECT shipping_mode, sum(days_actual), sum(days_scheduled), sum(days_actual)/sum(days_scheduled) FROM orders AS o

JOIN customers as c
ON o.customer_id = c.id

JOIN shipping as s
ON o.id = s.order_id

GROUP BY shipping_mode


ORDER BY quantity DESC, late_delivery_risk

LIMIT 10
"""

# Call the function to run the query and display results
result = run_query_to_df(cursor, query)
result

,shipping_mode,sum(days_actual),sum(days_scheduled),sum(days_actual)/sum(days_scheduled)
0,First Class,18804.0,9402.0,2.000000
1,Second Class,47540.0,23772.0,1.999832
2,Same Day,1589.0,0.0,NaN
3,Standard Class,145867.0,146152.0,0.998050


In [124]:
query = """
SELECT
    substr(
        order_date,
        instr(order_date, '/') +
        instr(substr(order_date, instr(order_date, '/') + 1), '/') + 1,
        4
    ) || '-' ||
    printf(
        '%02d',
        CAST(substr(order_date, 1, instr(order_date, '/') - 1) AS INTEGER)
    ) AS year_month,

    SUM(late_delivery_risk), COUNT(*) AS total_orders,
    ROUND(SUM(late_delivery_risk)*1.0/COUNT(*),2) AS late_risk

FROM orders AS o

JOIN shipping as s
ON s.order_id = o.id

GROUP BY year_month
ORDER BY year_month;
"""
# Call the function to run the query and display results
result = run_query_to_df(cursor, query)
result

,year_month,SUM(late_delivery_risk),total_orders,late_risk
0,2015-01,904,1679,0.54
1,2015-02,821,1496,0.55
2,2015-03,921,1682,0.55
3,2015-04,876,1619,0.54
4,2015-05,912,1686,0.54
5,2015-06,885,1633,0.54
6,2015-07,921,1670,0.55
7,2015-08,920,1675,0.55
8,2015-09,892,1613,0.55
9,2015-10,913,1670,0.55
